# Geometry-based Discovery of Calcium Battery Cathodes Accelerated by Foundational Machine-Learned Models
  #### Dereje Bekele Tekliye, Department of Materials Engineering, Indian Institute of Science 


The development of this notebook and analysis pipeline was assisted by AI tools, including Claude Pro (Anthropic, Claude Opus 4.7) and ChatGPT Pro (OpenAI). These tools were used to support code development, debugging, and workflow automation. All scientific decisions, validation, and interpretation of results remain the responsibility of the authors.


This notebook implements a six-criterion screening pipeline to identify promising Ca-ion battery
cathode materials using data from the Materials Project (MP) combined with machine-learning
interatomic potentials (MACE-MP-0, ORB-v3) and a transfer-learning (TL) migration-barrier model.

**Screening summary:**

| Step | Criterion                                                              | In → Out        |
|------|------------------------------------------------------------------------|-----------------|
| 1    | Ca Voronoi polyhedra volume range                                      | 52,945 → 5,946  |
| 2    | Charge neutrality (discharged & charged)                               | 5,946 → 1,581   |
| 3    | Exclusion of mobile-cation-containing compounds                        | 1,581 → 1,136   |
| 4    | Thermodynamic stability & voltage window                               | 1,136 → 433     |
| 5    | Curation (duplicates, MP overlap, non-viable pathways)                 | 433 → 221       |
| 6    | Ca-mobility analysis                                                   | 221 → 37        |

**Reproducibility note** — every path is derived from a single `PROJECT_ROOT` variable defined in
Section 1.2 below; every CSV name is registered in the `CSV` dict in Section 1.3. To reproduce
this work on a different machine, edit only those two blocks.


# 1. Imports & Configuration


## 1.1 Imports


In [ ]:
# --- Standard library ---
import os, glob, re, csv, gc, io, json, logging, time
from pathlib import Path
from itertools import combinations
from dataclasses import dataclass, field
from typing import Optional, Sequence, Union, Dict, Iterable, Tuple, List

import numpy as np
import pandas as pd

# --- Materials Project ---
from mp_api.client import MPRester
from mp_api.client.core.client import MPRestError

# --- Pymatgen core ---
from pymatgen.core import Element, Composition, Structure
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp import Poscar
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.transformations.standard_transformations import (
    OxidationStateDecorationTransformation,
    AutoOxiStateDecorationTransformation,
)
from pymatgen.analysis.energy_models import EnergyModel, EwaldElectrostaticModel
from pymatgen.analysis.structure_matcher import StructureMatcher
from pymatgen.analysis.local_env import VoronoiNN
from pymatgen.analysis.defects.utils import TopographyAnalyzer, remove_collisions
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry
from pymatgen.entries.computed_entries import ComputedEntry
from pymatgen.entries.compatibility import MaterialsProject2020Compatibility

# --- ASE / MACE / ORB (uncomment for Criterion 4 & 6) ---
# from ase.io import read, write
# from ase.optimize import QuasiNewton, BFGS
# from ase.filters import UnitCellFilter
# from ase.mep.neb import NEB
# from ase.constraints import FixAtoms
# from mace.calculators import mace_mp
# from orb_models.forcefield import pretrained
# from orb_models.forcefield.calculator import ORBCalculator
# import torch

# --- Transfer Learning (JARVIS) ---
# from jarvis.core.atoms import Atoms as JarvisAtoms

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)


## 1.2 Project paths

All paths are derived from a single `PROJECT_ROOT`. To reproduce on a different machine,
set the `CA_CATHODE_PROJECT_ROOT` environment variable or edit `PROJECT_ROOT` below.


In [ ]:
# --- Single editable root: change this (or set CA_CATHODE_PROJECT_ROOT) ---
PROJECT_ROOT = Path(os.environ.get("CA_CATHODE_PROJECT_ROOT", Path.cwd())).resolve()

# --- Derived paths ---
CSV_DIR         = PROJECT_ROOT / "csv"
STRUCTURES_DIR  = PROJECT_ROOT / "structures"   # {mp-id}_{formula}/{discharged,charged}/POSCAR
NEB_BASE_DIR    = STRUCTURES_DIR                # NEB sub-folders live under each material
MODELS_DIR      = PROJECT_ROOT / "models"
TL_DIR          = PROJECT_ROOT / "tl_model"
TL_INTERP_DIR   = TL_DIR / "interpolated"
TL_MP_MATCH_DIR = TL_DIR / "mp_matching"
TL_NEB_DIR      = STRUCTURES_DIR                # same NEB-structure folder

# --- Model files ---
MACE_MODEL_PATH = MODELS_DIR / "2024-01-07-mace-128-L2_epoch-199.model"

# --- Single-file artefacts ---
TL_STRUCTURE_JSON = TL_DIR / "structure_list.json"
MATERIAL_IDS_JSON = PROJECT_ROOT / "candidate_material_ids.json"

# Create output directories we write into
for d in (CSV_DIR, STRUCTURES_DIR, MODELS_DIR, TL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT    = {PROJECT_ROOT}")
print(f"CSV_DIR         = {CSV_DIR}")
print(f"STRUCTURES_DIR  = {STRUCTURES_DIR}")


## 1.3 CSV registry

A single dict maps every screening step to its CSV file. Cells later in the notebook only
reference `CSV[<key>]` — never a hard-coded filename. To rename outputs, edit this block only.


In [ ]:
CSV: Dict[str, Path] = {
    # ---- Criterion 1: Ca Voronoi polyhedra volume range ------------------
    "vpv_candidates":      CSV_DIR / "01_ca_vpv_candidates.csv",         # 5946
    "vpv_non_candidates":  CSV_DIR / "01_ca_vpv_non_candidates.csv",

    # ---- Criterion 2: Charge neutrality -----------------------------------
    "neutrality_check":    CSV_DIR / "02_ca_charge_neutrality_check.csv", # with neutrality cols
    "charge_neutral":      CSV_DIR / "02_ca_charge_neutral.csv",          # 1581

    # ---- Criterion 3: Mobile-cation exclusion -----------------------------
    "with_mobile":         CSV_DIR / "03_ca_with_mobile_cations.csv",
    "no_mobile":           CSV_DIR / "03_ca_no_mobile_cations.csv",       # 1136

    # ---- Criterion 4: Thermodynamic stability & voltage -------------------
    "mace_energy":         CSV_DIR / "04a_ca_mace_energies.csv",
    "mace_chgform":        CSV_DIR / "04b_ca_mace_charged_formula.csv",
    "mace_fu":             CSV_DIR / "04c_ca_mace_with_fu.csv",
    "mace_corrected":      CSV_DIR / "04d_ca_mace_corrected.csv",
    "mace_clean":          CSV_DIR / "04e_ca_mace_corrected_clean.csv",
    "mace_rejected":       CSV_DIR / "04e_ca_mace_corrected_rejected.csv",
    "e_above_hull":        CSV_DIR / "04f_ca_e_above_hull.csv",
    "e_hull_voltage":      CSV_DIR / "04g_ca_e_above_hull_voltage.csv",
    "meta_stable":         CSV_DIR / "04h_ca_meta_stable.csv",
    "stability_voltage":   CSV_DIR / "04_ca_stability_voltage.csv",       # 433

    # ---- Criterion 5: Curation --------------------------------------------
    "duplicates_removed":  CSV_DIR / "05a_ca_duplicates_removed.csv",
    "mp_matched":          CSV_DIR / "05b_ca_mp_matched.csv",
    "mp_novel":            CSV_DIR / "05c_ca_mp_novel.csv",
    "curated":             CSV_DIR / "05_ca_curated.csv",                 # 221 (input to NEB)
    "curated_for_neb":     CSV_DIR / "05_ca_curated_for_neb.csv",         # subset with viable paths

    # ---- Criterion 6: Ca mobility (NEB / TL) ------------------------------
    "mace_neb":            CSV_DIR / "06a_ca_mace_neb.csv",
    "mace_neb_labels":     CSV_DIR / "06a_ca_mace_neb_labels.csv",
    "mace_orbv3_labels":   CSV_DIR / "06b_ca_mace_orbv3_labels.csv",
    "not_nan":             CSV_DIR / "06c_ca_mace_orbv3_not_nan.csv",
    "with_tl":             CSV_DIR / "06d_ca_with_tl.csv",
    "candidates_flag":     CSV_DIR / "06e_ca_candidates_flagged.csv",
    "final_candidates":    CSV_DIR / "06f_ca_final_candidates.csv",
    "with_spacegroup":     CSV_DIR / "06_ca_with_spacegroup.csv",         # 37
    "dft_validation":      CSV_DIR / "06_ca_dft_validation.csv",
}

# Display the registered names
for k, v in CSV.items():
    print(f"  {k:22s} -> {v.name}")


## 1.4 Screening parameters


In [ ]:
# --- Element lists used throughout the screening ---
TRANSITION_METALS: List[str] = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Nb", "Mo"]
ANIONS:            List[str] = ["N", "O", "F",
                                "P", "S", "Cl",
                                     "Se", "Br",
                                     "Te", "I"]

# Sets of elements NOT allowed to be replaced by Ca during the VPV step
TRANS_METAL_SET = set(TRANSITION_METALS)
P_BLOCK_SET     = {
    "B",  "C",  "N",  "O",  "F",
    "Si", "P",  "S",  "Cl",
                "As", "Se", "Br",
                      "Te", "I",
                            "At",
}

# --- VPV target & tolerance (from Ca-containing reference compounds) ---
TARGET_VPV = 13.864317863575002          # median Voronoi polyhedron volume of Ca sites
VPV_RANGE  = (13.17110197039625,         # median * (1 - 0.05)
              14.557533756753752)        # median * (1 + 0.05)

# --- Voltage window & stability cut-off (Criterion 4) ---
VOLTAGE_WINDOW   = (2.0, 4.5)            # V
E_HULL_THRESHOLD = 0.100                 # eV / atom

# --- Ca-mobility threshold (Criterion 6) ---
NEB_BARRIER_THRESHOLD_meV = 1000.0       # meV

# --- Materials Project API key ---
# Replace with your own key, or set the MP_API_KEY environment variable
MP_API_KEY = os.environ.get("MP_API_KEY", "your-api-key")
api_key    = MP_API_KEY  # backwards-compat alias used by some helpers

# --- DFT energy of Ca metal (for voltage calculation; 4 Ca atoms per reference POSCAR) ---
E_CA_METAL_PER_ATOM = -7.924198 / 4

# --- TL (transfer-learning) normalization parameters ---
# These come from the dataset used to train the ALIGNN model — keep in sync.
TL_MEAN = None   # <-- set to training-set mean of barriers (eV)
TL_STD  = None   # <-- set to training-set std  of barriers (eV)
TL_MIN  = None   # <-- set to training-set min  of barriers (eV)
TL_MAX  = None   # <-- set to training-set max  of barriers (eV)


## 1.5 Helper functions

Shared utilities used by the Criterion-1 (VPV) screening and several downstream steps.


In [ ]:
def search_by_element(mpr, element, fields, retries=3, delay=5):
    """Search MP for compounds containing `element`, retrying transient failures."""
    for attempt in range(1, retries + 1):
        try:
            return mpr.materials.summary.search(elements=[element], fields=fields)
        except MPRestError as e:
            print(f"Attempt {attempt} for element {element} failed: {e}")
            if attempt < retries:
                time.sleep(delay)
            else:
                raise


def remove_collisions_custom(fcoords, structure, min_dist=1):
    """Filter candidate fractional coords that are too close to each other."""
    filtered = []
    for fc in fcoords:
        if all(np.linalg.norm(np.array(fc) - np.array(other)) >= min_dist for other in filtered):
            filtered.append(fc)
    return np.array(filtered)


# Skipped-compound bookkeeping (mutated by `process_compound`)
skipped_count: int = 0
skipped_ids:   List[str] = []


def interstitial_sites(compound, mpr, structure=None):
    """
    Compute interstitial (void) sites with TopographyAnalyzer, remove collisions,
    and decorate the structure with dummy "X" species at each void.
    """
    if structure is None:
        structure = mpr.get_structure_by_material_id(compound.material_id)

    present_elements = list(structure.composition.elements)
    framework_ions = [el for el in present_elements if el.symbol in ANIONS]
    cations        = [el for el in present_elements if el not in framework_ions]

    if not cations:
        print(f"No cations found in compound {compound.material_id}; returning original structure.")
        return structure

    try:
        inter_sites = TopographyAnalyzer(
            structure,
            framework_ions=framework_ions,
            cations=cations,
            image_tol=0.0001,
            max_cell_range=1,
            check_volume=False,
            clustering_tol=1,
            min_dist=2.5,
            constrained_c_frac=0.5,
        )
    except Exception as e:
        print(f"Error initializing TopographyAnalyzer for {compound.material_id}: {e}")
        return structure

    try:
        labeled_sites = inter_sites.labeled_sites
    except Exception as e:
        print(f"Error retrieving labeled_sites for {compound.material_id}: {e}")
        return structure

    if not labeled_sites:
        print(f"No interstitial sites found for {compound.material_id}.")
        return structure

    try:
        fcoords = np.atleast_2d(np.array([site for site, _ in labeled_sites]))
    except Exception as e:
        print(f"Error processing void coordinates for {compound.material_id}: {e}")
        return structure

    if fcoords.size == 0 or fcoords.shape[1] != 3:
        print(f"Empty/malformed void coords for {compound.material_id}.")
        return structure

    try:
        filtered_fcoords = remove_collisions(
            fcoords=fcoords, structure=inter_sites.structure, min_dist=1
        )
    except Exception as e:
        print(f"Error in remove_collisions for {compound.material_id}: {e}")
        return structure

    modified_structure = Structure.from_sites(inter_sites.structure)
    for coord in filtered_fcoords:
        modified_structure.append("X", coord)
    return modified_structure


def voronoi_polyhedra_for_structure(structure):
    """
    Average Voronoi polyhedron volume (vpv) per element, ignoring anions and selected
    transition metals.
    """
    volumes_by_element: Dict[str, list] = {}
    vnn = VoronoiNN()
    ignore = set(ANIONS) | TRANS_METAL_SET

    for i, site in enumerate(structure):
        if site.species_string in ignore:
            continue
        try:
            polyhedra = vnn.get_voronoi_polyhedra(structure, i)
            if not polyhedra:
                print(f"No polyhedra data for site {i}.")
                continue
            vol = sum(facet.get("volume", 0) for facet in polyhedra.values())
            volumes_by_element.setdefault(site.species_string, []).append(vol)
        except ValueError as e:
            print(f"Skipping site {i} due to Voronoi error: {e}")
            continue

    return {el: (sum(v) / len(v) if v else None) for el, v in volumes_by_element.items()}


def get_candidate_element(vol_dict, vpv_range, target_vpv, trans_metal_set, p_block_set):
    """
    From the per-element vpv dict, pick the element (excluding "X", transition metals
    and p-block) whose vpv is closest to `target_vpv` and inside `vpv_range`. Falls
    back to the dummy "X" entry if no real element qualifies.
    """
    candidate_list = []
    for elem, vpv in vol_dict.items():
        if elem == "X":
            continue
        if elem in trans_metal_set or elem in p_block_set:
            continue
        if vpv is not None and vpv_range[0] <= vpv <= vpv_range[1]:
            candidate_list.append((elem, vpv))
    if candidate_list:
        return min(candidate_list, key=lambda x: abs(x[1] - target_vpv))
    if "X" in vol_dict:
        vpv_x = vol_dict["X"]
        if vpv_x is not None and vpv_range[0] <= vpv_x <= vpv_range[1]:
            return ("X", vpv_x)
    return None


def update_composition_with_ca(structure, candidate):
    """Substitute Ca for `candidate` element (and drop dummy X) in the composition."""
    new_comp = structure.composition.copy()
    cand_elem, _ = candidate
    amt = new_comp.get(cand_elem, 0)
    if amt > 0:
        new_comp = new_comp - Composition({cand_elem: amt}) + Composition({"Ca": amt})
    if "X" in new_comp:
        new_comp = new_comp - Composition({"X": new_comp.get("X", 0)})
    return new_comp


def process_compound(compound, mpr):
    """
    Full per-compound pipeline:
      1. Retrieve structure from MP.
      2. Compute interstitial sites and Voronoi volumes.
      3. Select a candidate element to swap for Ca.
      4. Build discharged (Ca-substituted) and charged (Ca-removed) structures.
    """
    global skipped_count, skipped_ids
    try:
        orig_struct = mpr.get_structure_by_material_id(compound.material_id)
        mod_struct  = interstitial_sites(compound, mpr, orig_struct)
        used_struct = mod_struct if "X" in mod_struct.composition else orig_struct

        vol_dict  = voronoi_polyhedra_for_structure(used_struct)
        candidate = get_candidate_element(
            vol_dict, VPV_RANGE, TARGET_VPV, TRANS_METAL_SET, P_BLOCK_SET
        )

        if candidate:
            cand_elem, vpv = candidate
            discharged = used_struct.copy()
            discharged.replace_species({cand_elem: "Ca"})
            discharged.remove_species(["X"])

            charged = discharged.copy()
            charged.remove_species(["Ca"])

            return True, {
                "material_id":          compound.material_id,
                "original_formula":     used_struct.composition.reduced_formula,
                "new_formula":          discharged.composition.reduced_formula,
                "replaced_element":     cand_elem,
                "vpv":                  vpv,
                "discharged_structure": discharged,
                "charged_structure":    charged,
            }
        return False, {
            "material_id":          compound.material_id,
            "original_formula":     used_struct.composition.reduced_formula,
            "volumes_by_element":   vol_dict,
            "discharged_structure": None,
            "charged_structure":    None,
        }
    except Exception as e:
        skipped_count += 1
        skipped_ids.append(compound.material_id)
        print(f"Skipping {compound.material_id}: {e}")
        return False, {"material_id": compound.material_id, "error": str(e)}


# 2. Criterion 1 — Ca Voronoi Polyhedra Volume

Query MP for compounds containing at least one transition metal *and* one anion (excluding any
Ca-bearing entries), then keep those whose voronoi-polyhedra volume (VPV) for one of the
non-TM / non-p-block elements falls inside `VPV_RANGE`. The chosen element is later substituted
by Ca to form the discharged structure.


## 2.1 Query MP and run VPV screen


In [ ]:
candidate_compounds:     List[dict] = []
non_candidate_compounds: List[dict] = []

with MPRester(MP_API_KEY) as mpr:
    fields = ["material_id", "formula_pretty", "elements"]

    # Query compounds containing any transition metal.
    tm_compounds: Dict[str, object] = {}
    for el in TRANSITION_METALS:
        for entry in search_by_element(mpr, el, fields):
            tm_compounds[entry.material_id] = entry

    # Query compounds containing any anion.
    anion_compounds: Dict[str, object] = {}
    for el in ANIONS:
        for entry in search_by_element(mpr, el, fields):
            anion_compounds[entry.material_id] = entry

    common_ids = set(tm_compounds.keys()).intersection(anion_compounds.keys())
    # Exclude entries whose elements already include Ca
    compounds = [
        tm_compounds[mid] for mid in common_ids
        if "Ca" not in str(tm_compounds[mid].elements)
    ]
    print(f"Found {len(compounds)} compounds with TM + anion (excluding Ca).")

    for compound in compounds:
        is_candidate, result = process_compound(compound, mpr)
        if is_candidate:
            candidate_compounds.append(result)
            print(f"  {result['material_id']}: replaced {result['replaced_element']} "
                  f"(vpv {result['vpv']:.4f}) with Ca")
        else:
            non_candidate_compounds.append(result)


## 2.2 Save candidate / non-candidate lists


In [ ]:
df_candidates     = pd.DataFrame(candidate_compounds)
df_non_candidates = pd.DataFrame(non_candidate_compounds)

df_candidates.to_csv(CSV["vpv_candidates"], index=False)
df_non_candidates.to_csv(CSV["vpv_non_candidates"], index=False)
print(f"VPV candidates : {len(df_candidates):>5d} -> {CSV['vpv_candidates'].name}")
print(f"Non-candidates : {len(df_non_candidates):>5d} -> {CSV['vpv_non_candidates'].name}")


# 3. Criterion 2 — Charge Neutrality

Check that both the discharged (Ca-bearing) and charged (Ca-removed) formulas admit a
chemically reasonable, charge-neutral oxidation-state assignment subject to:
- transition-metal oxidation ∈ allowed ranges (V: 2–5, others: 2–4),
- p-block oxidation ∈ allowed integer states,
- F = −1 when present, and Si/P/S forced to +4/+5/+6 when O is present.


## 3.1 Build the charged formula by stripping Ca


In [ ]:
df_candidates = pd.read_csv(CSV["vpv_candidates"])


def remove_ca(formula: str) -> str:
    """Return `formula` with all Ca removed (reduced)."""
    comp = Composition(formula)
    el_amt = comp.get_el_amt_dict()
    if "Ca" not in el_amt:
        return formula
    el_amt.pop("Ca")
    if not el_amt:
        return ""
    return Composition(el_amt).formula


df_candidates["charged_formula"] = df_candidates["new_formula"].apply(remove_ca)
# Move the new column right after `new_formula`
col_index = df_candidates.columns.get_loc("new_formula")
df_candidates.insert(col_index + 1, "charged_formula", df_candidates.pop("charged_formula"))


## 3.2 Apply oxidation-state / charge-neutrality checks


In [ ]:
# --- Allowed transition-metal oxidation ranges ---
TM_OXI_RANGES: Dict[str, Tuple[int, int]] = {m: (2, 4) for m in
                                              ["Ti", "Cr", "Mn", "Fe", "Co", "Ni", "Nb", "Mo"]}
TM_OXI_RANGES["V"] = (2, 5)
TM_LIST = list(TM_OXI_RANGES.keys())

# --- Allowed p-block oxidation states ---
P_BLOCK_OXI: Dict[str, List[int]] = {
    "B":  [3],           "Al": [3],              "Ga": [3],           "In": [3],
    "C":  [-4, -3, -2, -1, 1, 2, 3, 4],
    "Si": [-4, 4],       "Ge": [-4, 2, 4],       "Sn": [-4, 2, 4],
    "N":  [-3, 3, 5],    "P":  [-3, 3, 5],       "As": [-3, 3, 5],    "Sb": [-3, 3, 5], "Bi": [3],
    "O":  [-2],          "S":  [-2, 2, 4, 6],    "Se": [-2, 2, 4, 6], "Te": [-2, 2, 4, 6],
    "F":  [-1],          "Cl": [-1, 1, 3, 5, 7], "Br": [-1, 1, 3, 5], "I":  [-1, 1, 3, 5, 7],
}

NEUTRALITY_TOL = 1e-6


def is_charge_neutral(formula: str, tol: float = NEUTRALITY_TOL):
    """
    Try every oxi_state_guesses() assignment from pymatgen. Return (neutral, state)
    for the first one that satisfies:
      1) net charge = 0,
      2) TM oxidation ∈ TM_OXI_RANGES,
      3) p-block oxidation is integer,
      4) p-block oxidation ∈ P_BLOCK_OXI[element],
      5) F == -1 if present,
      6) if O present: Si=+4, P=+5, S=+6,
      7) if >1 p-block (excluding F), least-EN must be positive, most-EN negative.
    """
    comp = Composition(formula)
    for state in comp.oxi_state_guesses():
        # 1) net zero charge
        if abs(sum(comp[el] * ox for el, ox in state.items())) > tol:
            continue
        # 2) TM ranges
        if any(
            el in TM_OXI_RANGES and (
                state[el] < TM_OXI_RANGES[el][0] - tol
                or state[el] > TM_OXI_RANGES[el][1] + tol
            )
            for el in state
        ):
            continue
        # 3) p-block must be integer
        p_elems = [el for el in state if el in P_BLOCK_OXI]
        if any(abs(state[el] - round(state[el])) > tol for el in p_elems):
            continue
        # 4) explicit allowed p-block states
        if any(round(state[el]) not in P_BLOCK_OXI[el] for el in p_elems):
            continue
        # 5) F exception
        if "F" in state and abs(state["F"] + 1) > tol:
            continue
        # 6) Si/P/S forced when O present
        if "O" in state:
            fixed = {"Si": 4, "P": 5, "S": 6}
            if any(el in state and abs(state[el] - req) > tol for el, req in fixed.items()):
                continue
        # 7) relative p-block sign rule (skip if F present)
        if "F" not in state and len(p_elems) > 1:
            sorted_p = sorted(p_elems, key=lambda el: Element(el).X or 0)
            least, most = sorted_p[0], sorted_p[-1]
            if not (state[least] > 0 and state[most] < 0):
                continue
        return True, state
    return False, None


def make_assign_oxidation_states(formula_col: str, prefix: str):
    """
    Return a row-wise function that adds three columns:
      {prefix} Charge Neutral
      {prefix} Oxidation Assignment
      {prefix} TM Oxidation Assignment
    """
    def _assign(row):
        formula = row.get(formula_col)
        if pd.isna(formula) or not formula:
            return pd.Series({
                f"{prefix} Charge Neutral":          False,
                f"{prefix} Oxidation Assignment":    None,
                f"{prefix} TM Oxidation Assignment": None,
            })
        neutral, full_assignment = is_charge_neutral(formula)
        ox_assign_str = str(full_assignment) if full_assignment else None
        tm_assign = None
        if full_assignment:
            tm_only = {el: full_assignment[el] for el in full_assignment if el in TM_LIST}
            tm_assign = str(tm_only) if tm_only else None
        return pd.Series({
            f"{prefix} Charge Neutral":          neutral,
            f"{prefix} Oxidation Assignment":    ox_assign_str,
            f"{prefix} TM Oxidation Assignment": tm_assign,
        })
    return _assign


assign_new     = make_assign_oxidation_states("new_formula",     "discharged")
assign_charged = make_assign_oxidation_states("charged_formula", "charged")

new_cols     = df_candidates.apply(assign_new,     axis=1)
charged_cols = df_candidates.apply(assign_charged, axis=1)
df_candidates = pd.concat([df_candidates, new_cols, charged_cols], axis=1)

df_candidates.to_csv(CSV["neutrality_check"], index=False)
print(f"Saved per-row neutrality check ({len(df_candidates)} rows) -> {CSV['neutrality_check'].name}")


## 3.3 Keep rows neutral in both phases


In [ ]:
df_neutrality = pd.read_csv(CSV["neutrality_check"])

df_charge_neutral = df_neutrality[
    (df_neutrality["discharged Charge Neutral"] == True) &
    (df_neutrality["charged Charge Neutral"]    == True)
]

df_charge_neutral.to_csv(CSV["charge_neutral"], index=False)
print(f"Both-phases neutral: {len(df_charge_neutral)} -> {CSV['charge_neutral'].name}")


# 4. Criterion 3 — Exclusion of mobile-cation-containing compounds

Drop any compound whose discharged or charged oxidation assignment contains a competing mobile
cation (H, Li, Na, K, Mg, Zn). Those would complicate Ca-only intercalation analysis.


In [ ]:
df = pd.read_csv(CSV["charge_neutral"])


def _norm(s) -> str:
    """Strip + collapse whitespace + fix NBSP for robust column-name matching."""
    return re.sub(r"\s+", " ", str(s).replace("\u00a0", " ")).strip()


df.columns = [_norm(c) for c in df.columns]

# Find the discharged / charged oxidation-assignment columns robustly
cand_cols: List[str] = []
for probe in ["discharged Oxidation Assignment", "charged Oxidation Assignment"]:
    for c in df.columns:
        lc = _norm(c).lower()
        if "oxidation assignment" in lc and probe.split()[0] in lc:
            cand_cols.append(c)
            break

# Match {"H": | "Li": | ... } as a dict key in either column
MOBILE_PATTERN = re.compile(r"""[\"'](?:H|Li|Na|K|Mg|Zn)[\"']\s*:""")


def _has_mobile(x) -> bool:
    return bool(MOBILE_PATTERN.search("" if pd.isna(x) else str(x)))


mask_any = pd.Series(False, index=df.index)
for col in cand_cols:
    mask_any |= df[col].map(_has_mobile)

df_with_mobile = df[mask_any].copy()
df_no_mobile   = df[~mask_any].copy()

df_with_mobile.to_csv(CSV["with_mobile"], index=False)
df_no_mobile.to_csv(CSV["no_mobile"], index=False)

print(f"With mobile cations    : {len(df_with_mobile):>5d} -> {CSV['with_mobile'].name}")
print(f"Without mobile cations : {len(df_no_mobile):>5d} -> {CSV['no_mobile'].name}")


# 5. Criterion 4 — Thermodynamic Stability & Voltage

This section runs in seven sub-steps:

1. **5.1** Retrieve and save discharged / charged structures for relaxation.
2. **5.2** Geometry relaxation with MACE-MP-0 (total-energy evaluation).
3. **5.3** Normalize MACE energies to per-formula-unit.
4. **5.4** Apply MaterialsProject2020 GGA / GGA+U / anion corrections.
5. **5.5** Reject relaxations whose per-atom energy is physically implausible.
6. **5.6** Build a combined convex hull (MP + dataset) and compute E_above_hull.
7. **5.7** Compute the average intercalation voltage and apply the stability + voltage filter.


## 5.1 Retrieve structures for geometry relaxation

For every material that survived Criterion 3, save the discharged (Ca-substituted) and charged
(Ca-removed) POSCARs to `STRUCTURES_DIR / {mp-id}_{formula} / {discharged,charged} / POSCAR`.


In [ ]:
# Save the list of material IDs that passed Criterion 3 (used downstream)
df_no_mobile = pd.read_csv(CSV["no_mobile"])
mat_ids: List[str] = df_no_mobile["material_id"].tolist()
with open(MATERIAL_IDS_JSON, "w") as f:
    json.dump(mat_ids, f, indent=2)
print(f"Wrote {len(mat_ids)} material IDs -> {MATERIAL_IDS_JSON}")

# Re-query MP for those IDs and write discharged / charged POSCARs to STRUCTURES_DIR
with open(MATERIAL_IDS_JSON) as f:
    material_ids = set(json.load(f))

with MPRester(MP_API_KEY) as mpr:
    fields = ["material_id", "formula_pretty", "elements"]

    tm_compounds: Dict[str, object] = {}
    for el in TRANSITION_METALS:
        for entry in search_by_element(mpr, el, fields):
            tm_compounds[entry.material_id] = entry

    anion_compounds: Dict[str, object] = {}
    for el in ANIONS:
        for entry in search_by_element(mpr, el, fields):
            anion_compounds[entry.material_id] = entry

    common_ids = set(tm_compounds.keys()).intersection(anion_compounds.keys())
    compounds = [
        tm_compounds[mid] for mid in common_ids
        if "Ca" not in str(tm_compounds[mid].elements)
    ]

    for compound in compounds:
        if compound.material_id not in material_ids:
            continue

        is_candidate, result = process_compound(compound, mpr)
        if not is_candidate:
            print(f"{compound.material_id} not a candidate; skipping save.")
            continue

        new_formula = result["new_formula"]
        root_path   = STRUCTURES_DIR / f"{compound.material_id}_{new_formula}"

        for phase_key, phase_label in [("discharged_structure", "discharged"),
                                       ("charged_structure",    "charged")]:
            struct = result.get(phase_key)
            if struct is None:
                print(f"No `{phase_label}` struct for {compound.material_id}; skipping.")
                continue
            phase_dir = root_path / phase_label
            phase_dir.mkdir(parents=True, exist_ok=True)
            poscar_path = phase_dir / "POSCAR"
            Poscar(struct).write_file(str(poscar_path))
            print(f"Saved {phase_label} POSCAR for {compound.material_id} -> {poscar_path}")


## 5.2 Geometry relaxation and total-energy evaluation with MACE-MP-0

Uncomment the ASE / MACE / torch imports in Section 1.1 before running this cell.
Each compound's `discharged` and `charged` POSCAR is relaxed under a `UnitCellFilter`
(cell + ionic) using `QuasiNewton`. Results are checkpointed via `POSCAR_fin` and the
trajectory file, so re-running this cell skips compounds that are already relaxed.


In [ ]:
# Uncomment the ASE / MACE imports in Section 1.1 before running.
# from ase.io import read, write
# from ase.optimize import QuasiNewton
# from ase.filters import UnitCellFilter
# from mace.calculators import mace_mp

DEVICE = "cpu"   # set to "cuda" if a GPU is available

df = pd.read_csv(CSV["no_mobile"])

# Ensure energy columns exist
for phase in ("discharged", "charged"):
    col = f"{phase}_energy"
    if col not in df.columns:
        df[col] = pd.NA

for idx, row in df.iterrows():
    mp_id       = row["material_id"]
    new_formula = str(row.get("new_formula", "")).replace(" ", "")
    root_path   = STRUCTURES_DIR / f"{mp_id}_{new_formula}"

    for phase in ("discharged", "charged"):
        phase_dir      = root_path / phase
        poscar_orig    = phase_dir / "POSCAR"
        poscar_relaxed = phase_dir / "POSCAR_fin"
        traj           = phase_dir / f"{phase}_cell_relaxation.traj"

        # Re-use the existing relaxation if present
        if poscar_relaxed.exists():
            if traj.exists():
                try:
                    frames = read(str(traj), index=":")
                    energy = frames[-1].get_potential_energy()
                    df.at[idx, f"{phase}_energy"] = energy
                    print(f"{idx}: {mp_id} {phase}: existing energy = {energy:.6f} eV")
                except Exception as e:
                    print(f"Error reading {traj} for {mp_id} {phase}: {e}")
            else:
                print(f"{mp_id} {phase}: POSCAR_fin exists but trajectory missing; skipping.")
            continue

        if not poscar_orig.exists():
            print(f"Missing POSCAR: {poscar_orig}; skipping {phase} for {mp_id}.")
            continue

        # a) Read & attach calculator
        atoms = read(str(poscar_orig))
        atoms.calc = mace_mp(
            model=str(MACE_MODEL_PATH),
            default_dtype="float64",
            device=DEVICE,
        )

        # b) Relax cell + atomic positions
        print(f"relaxing index {idx} : {mp_id}")
        ucf = UnitCellFilter(atoms)
        opt = QuasiNewton(ucf, trajectory=str(traj))
        opt.run(fmax=0.05, steps=1000)

        # c) Write relaxed geometry to POSCAR_fin
        write(str(poscar_relaxed), atoms, format="vasp")
        print(f"{mp_id} {phase} relaxed -> {poscar_relaxed}")

        # d) Record final potential energy
        energy = atoms.get_potential_energy()
        df.at[idx, f"{phase}_energy"] = energy
        print(f"{mp_id} {phase} final energy = {energy:.6f} eV")

# Drop rows without a discharged energy and persist
df = df[~df["discharged_energy"].isna()]
df.to_csv(CSV["mace_energy"], index=False)
print(f"\nSaved {len(df)} rows -> {CSV['mace_energy'].name}")


## 5.3 Normalize MACE energy per formula unit

The MACE energy is total per simulation cell. To put both discharged and charged on the same
scale we:
1. Rebuild `charged_formula` as `new_formula` minus Ca (handles parentheses and counts).
2. Determine the number of formula units `fu_new` in each relaxed POSCAR.
3. Divide both energies by `fu_new` to get per-formula-unit values.


In [ ]:
def strip_ca(formula: str) -> str:
    """
    Remove Ca (with optional count) from a formula string.
    Matches 'Ca' NOT followed by a lowercase letter (so it does not eat Cd, Cl, ...)
    optionally followed by digits.
    """
    if not isinstance(formula, str):
        return formula
    return re.sub(r"Ca(?![a-z])\d*", "", formula)


df = pd.read_csv(CSV["mace_energy"])

if "new_formula" not in df.columns or "charged_formula" not in df.columns:
    raise SystemExit("CSV must contain both 'new_formula' and 'charged_formula' columns")

df["charged_formula"] = df["new_formula"].apply(strip_ca)
df.to_csv(CSV["mace_chgform"], index=False)
print(f"Wrote {CSV['mace_chgform'].name}  ({len(df)} rows)")

print("\nSample (new_formula -> charged_formula):")
for _, row in df.head(10).iterrows():
    print(f"  {row['new_formula']:30s} -> {row['charged_formula']}")


In [ ]:
def count_atoms_in_poscar(poscar_path: Path, exclude: str = "Ca") -> int:
    """
    Parse a VASP5 POSCAR and return the total atom count, excluding `exclude`.
    Element symbols on line 6, counts on line 7 (0-indexed: 5 and 6).
    """
    with poscar_path.open() as f:
        lines = f.readlines()
    elements = lines[5].split()
    counts   = [int(x) for x in lines[6].split()]
    if len(elements) != len(counts):
        raise ValueError(f"Element/count mismatch in {poscar_path}")
    return sum(c for e, c in zip(elements, counts) if e != exclude)


def count_atoms_in_formula(formula: str) -> int:
    """
    Total atom count for a chemical formula, supporting nested parentheses.
    Examples: 'CaFe2(AgO2)2' -> 1+2+2+4 = 9.
    """
    i, n = 0, len(formula)

    def parse(depth: int):
        nonlocal i
        total = 0
        while i < n:
            ch = formula[i]
            if ch == "(":
                i += 1
                inner = parse(depth + 1)
                m = re.match(r"\d+", formula[i:])
                if m:
                    mult = int(m.group()); i += len(m.group())
                else:
                    mult = 1
                total += inner * mult
            elif ch == ")":
                i += 1
                return total
            elif ch.isupper():
                sym = re.match(r"[A-Z][a-z]?", formula[i:])
                i += len(sym.group())
                cnt = re.match(r"\d+", formula[i:])
                if cnt:
                    total += int(cnt.group()); i += len(cnt.group())
                else:
                    total += 1
            else:
                i += 1
        return total

    return parse(0)


def find_material_folder(root: Path, material_id: str) -> Optional[Path]:
    """Find a folder under `root` whose name starts with '{material_id}_'."""
    matches = sorted(root.glob(f"{material_id}_*"))
    if not matches:
        return None
    if len(matches) > 1:
        print(f"  [warn] multiple folders for {material_id}, using first: {matches[0].name}")
    return matches[0]


# --- Apply ---
POSCAR_FOR_FU      = "POSCAR"        # one of: POSCAR, POSCAR_fin
SUBDIR_FOR_FU      = "discharged"
EXCLUDE_FOR_FU     = "Ca"

df = pd.read_csv(CSV["mace_chgform"])
fu_values: List[Optional[float]] = []

for _, row in df.iterrows():
    mid     = str(row["material_id"]).strip()
    charged = str(row["charged_formula"]).strip()

    folder = find_material_folder(STRUCTURES_DIR, mid)
    if folder is None:
        print(f"[{mid}] folder not found - skipping")
        fu_values.append(None); continue

    poscar = folder / SUBDIR_FOR_FU / POSCAR_FOR_FU
    if not poscar.exists():
        print(f"[{mid}] {poscar} not found - skipping")
        fu_values.append(None); continue

    try:
        n_poscar  = count_atoms_in_poscar(poscar, exclude=EXCLUDE_FOR_FU)
        n_charged = count_atoms_in_formula(charged)
        if n_charged == 0:
            print(f"[{mid}] charged_formula '{charged}' parsed to 0 atoms - skipping")
            fu_values.append(None); continue
        fu = n_poscar / n_charged
        fu_values.append(fu)
        print(f"[{mid}] poscar(no {EXCLUDE_FOR_FU})={n_poscar}  "
              f"charged({charged})={n_charged}  fu_new={fu:g}")
    except Exception as e:
        print(f"[{mid}] error: {e}")
        fu_values.append(None)

df["fu_new"] = fu_values
df.to_csv(CSV["mace_fu"], index=False)
print(f"\nWrote {CSV['mace_fu'].name}  "
      f"({df['fu_new'].notna().sum()}/{len(df)} rows populated)")


In [ ]:
df = pd.read_csv(CSV["mace_fu"])

required = {"discharged_energy", "charged_energy", "fu_new"}
missing = required - set(df.columns)
if missing:
    raise SystemExit(f"Missing required columns: {missing}")

df["fu_new"]            = pd.to_numeric(df["fu_new"],            errors="coerce")
df["discharged_energy"] = pd.to_numeric(df["discharged_energy"], errors="coerce")
df["charged_energy"]    = pd.to_numeric(df["charged_energy"],    errors="coerce")

# Avoid divide-by-zero: rows with fu_new == 0 or NaN -> NaN result
fu = df["fu_new"].where((df["fu_new"] != 0) & df["fu_new"].notna())

df["discharged_energy_per_fu"] = df["discharged_energy"] / fu
df["charged_energy_per_fu"]    = df["charged_energy"]    / fu

df.to_csv(CSV["mace_fu"], index=False)
n_bad = df["discharged_energy_per_fu"].isna().sum()
print(f"Updated {CSV['mace_fu'].name}  ({len(df)} rows, {n_bad} NaN per-fu values)")
print("\nPreview:")
print(df[["material_id", "discharged_energy", "charged_energy",
          "fu_new", "discharged_energy_per_fu", "charged_energy_per_fu"]]
      .head().to_string(index=False))


## 5.4 Apply MaterialsProject2020 corrections to MACE energies

`MaterialsProject2020Compatibility(compat_type="Advanced")` applies GGA / GGA+U and anion
corrections so the MACE-derived energies are on the same scale as MP-tabulated formation
energies. The Hubbard-U values follow `MPRelaxSet`.


In [ ]:
# Hubbard-U values used by MPRelaxSet
U_MAP = {
    "Ti": 0.0,
    "V":  3.25,
    "Cr": 3.7,
    "Mn": 3.9,
    "Fe": 5.3,
    "Co": 3.32,
    "Ni": 6.2,
    "W":  6.2,
    "Nb": 0.0,
    "Mo": 4.38,
}

compat = MaterialsProject2020Compatibility(
    compat_type="Advanced",
    correct_peroxide=False,
    strict_anions="require_bound",
    check_potcar=False,
    check_potcar_hash=False,
)


def mp2020_correct_advanced(formula: str, E_per_fu: float) -> Tuple[float, float]:
    """Apply MP2020 Advanced correction to one (formula, total-energy-per-FU)."""
    comp    = Composition(formula)
    E_total = E_per_fu  # already total per FU

    entry = ComputedEntry(
        comp,
        E_total,
        parameters={
            "run_type": "GGA+U",
            "hubbards": {
                el.symbol: U_MAP.get(el.symbol, 0.0)
                for el in comp.elements
                if U_MAP.get(el.symbol, 0.0) > 0
            },
        },
    )
    processed = compat.process_entries([entry], clean=True)
    if processed:
        ce = processed[0]
        return ce.energy, ce.correction
    return E_total, 0.0


df = pd.read_csv(CSV["mace_fu"])

dis_results = df.apply(
    lambda r: pd.Series(mp2020_correct_advanced(r["new_formula"],
                                                r["discharged_energy_per_fu"])),
    axis=1,
)
dis_results.columns = ["discharged_energy_fu_corrected", "_disch_correction"]

chg_results = df.apply(
    lambda r: pd.Series(mp2020_correct_advanced(r["charged_formula"],
                                                r["charged_energy_per_fu"])),
    axis=1,
)
chg_results.columns = ["charged_energy_fu_corrected", "applied_correction"]

df = pd.concat([df, dis_results, chg_results], axis=1)
# Drop the duplicate correction column (the per-formula-unit correction is the same scalar
# for both phases for our anion sets)
df.drop("_disch_correction", axis=1, inplace=True)

df.to_csv(CSV["mace_corrected"], index=False)
print(f"Wrote MP2020-corrected energies -> {CSV['mace_corrected'].name}  ({len(df)} rows)")


## 5.5 Reject relaxations with unphysical energies

Real MACE-MP-0 corrected energies sit between roughly −1 (light elements) and −15 eV/atom
(heavy / well-relaxed TM compounds). Anything outside `[-20, 0]` eV/atom signals a failed
relaxation (e.g. divergent energy from an unphysical H-substitution). Rejected rows are
written to a separate CSV for inspection.


In [ ]:
MIN_PER_ATOM = -20.0
MAX_PER_ATOM =   0.0


def _per_atom(formula, energy) -> float:
    """Energy per atom from total per-FU energy."""
    if pd.isna(formula) or pd.isna(energy):
        return float("nan")
    try:
        n = Composition(formula).num_atoms
        return float(energy) / n if n > 0 else float("nan")
    except Exception:
        return float("nan")


df   = pd.read_csv(CSV["mace_corrected"])
n_in = len(df)

df["_dis_per_atom"] = df.apply(
    lambda r: _per_atom(r["new_formula"],     r["discharged_energy_fu_corrected"]), axis=1)
df["_chg_per_atom"] = df.apply(
    lambda r: _per_atom(r["charged_formula"], r["charged_energy_fu_corrected"]),    axis=1)

bad_dis  = (df["_dis_per_atom"] < MIN_PER_ATOM) | (df["_dis_per_atom"] > MAX_PER_ATOM)
bad_chg  = (df["_chg_per_atom"] < MIN_PER_ATOM) | (df["_chg_per_atom"] > MAX_PER_ATOM)
bad_mask = bad_dis | bad_chg

df_bad  = df[bad_mask].copy()
df_good = df[~bad_mask].drop(columns=["_dis_per_atom", "_chg_per_atom"])

print(f"Loaded {n_in} rows from {CSV['mace_corrected'].name}")
print(f"  Sane:     {len(df_good)}")
print(f"  Rejected: {len(df_bad)}")

if len(df_bad):
    print("\nRejected rows (per-atom energies in eV/atom):")
    cols = ["material_id", "new_formula", "charged_formula",
            "replaced_element", "_dis_per_atom", "_chg_per_atom"]
    for _, row in df_bad[cols].iterrows():
        d, c = row["_dis_per_atom"], row["_chg_per_atom"]
        df_flag = " <-" if d < MIN_PER_ATOM or d > MAX_PER_ATOM else ""
        cf_flag = " <-" if c < MIN_PER_ATOM or c > MAX_PER_ATOM else ""
        print(f"  {row['material_id']:13s}  {row['new_formula']:25s} / "
              f"{str(row['charged_formula']):20s}  "
              f"replaced={str(row['replaced_element']):4s}  "
              f"dis={d:+10.2f}{df_flag}  chg={c:+10.2f}{cf_flag}")
    if "replaced_element" in df_bad.columns:
        print("\nReplaced element among rejected rows:")
        print(df_bad["replaced_element"].value_counts().to_string())

df_good.to_csv(CSV["mace_clean"], index=False)
df_bad.drop(columns=["_dis_per_atom", "_chg_per_atom"]).to_csv(CSV["mace_rejected"], index=False)
print(f"\nWrote cleaned CSV   : {CSV['mace_clean'].name}    ({len(df_good)} rows)")
print(f"Wrote rejected CSV  : {CSV['mace_rejected'].name} ({len(df_bad)} rows)")


## 5.6 Convex-hull construction: compute energy above the hull

For each candidate (discharged and charged) we build a combined convex hull from:

  - **(a)** Materials Project entries in the same chemical system *and* all subsystems;
  - **(b)** all dataset candidates whose chemsys is a subset of the target.

The primary MP fetch path is `get_entries_in_chemsys` (uniform corrected-energy scale).
We fall back to `thermo.search` only when the primary call returns nothing — this happens
for a few rare-earth chemsys on some `mp_api` versions.


In [ ]:
# ---- Column-name conventions ----
F_DIS, F_CHG = "new_formula", "charged_formula"
E_DIS, E_CHG = "discharged_energy_fu_corrected", "charged_energy_fu_corrected"
H_DIS, H_CHG = "discharged_E_above_hull", "charged_E_above_hull"

PRINT_COMPETING_PHASES = True


def chemsys_of(formula: str) -> Tuple[str, ...]:
    return tuple(sorted({el.symbol for el in Composition(formula).elements}))


def make_pd_entry(formula: str, energy_total: float, name: str) -> PDEntry:
    return PDEntry(Composition(formula), float(energy_total), name=name)


def collect_dataset_entries(df: pd.DataFrame) -> Dict[tuple, list]:
    """Group dataset candidates by their chemsys."""
    grouped: Dict[tuple, list] = {}
    for idx, row in df.iterrows():
        for state, fcol, ecol in [("discharged", F_DIS, E_DIS),
                                  ("charged",   F_CHG, E_CHG)]:
            formula, energy = row[fcol], row[ecol]
            if pd.isna(formula) or pd.isna(energy):
                continue
            try:
                cs = chemsys_of(formula)
            except Exception as e:
                logger.warning(f"  [{row['material_id']}] bad formula '{formula}': {e}")
                continue
            entry = make_pd_entry(formula, float(energy),
                                  name=f"{row['material_id']}_{state}")
            grouped.setdefault(cs, []).append((idx, state, entry))
    return grouped


def gather_subsystem_dataset_entries(target_cs, own_by_cs):
    """All dataset entries whose chemsys is a subset of `target_cs`."""
    target_set = set(target_cs)
    out = []
    for cs_other, entries in own_by_cs.items():
        if set(cs_other).issubset(target_set):
            out.extend(entries)
    return out


def _entries_to_pd(docs) -> List[PDEntry]:
    """Convert ComputedEntries / ThermoDocs to PDEntries with corrected total energy."""
    out = []
    for d in docs:
        comp    = Composition(dict(d.composition))
        E_total = d.energy_per_atom * comp.num_atoms
        name    = getattr(d, "entry_id", None) or getattr(d, "material_id", "MP")
        out.append(PDEntry(comp, E_total, name=str(name)))
    return out


def fetch_mp_entries(mpr: MPRester, elems: Tuple[str, ...]) -> List[PDEntry]:
    """Primary: get_entries_in_chemsys; fallback: thermo.search."""
    cs_str = "-".join(sorted(elems))
    try:
        docs = mpr.get_entries_in_chemsys(list(elems))
        if docs:
            return _entries_to_pd(docs)
        logger.info(f"    get_entries_in_chemsys returned 0 for {cs_str}; "
                    f"falling back to thermo.search")
    except Exception as e:
        logger.warning(f"    get_entries_in_chemsys failed for {cs_str}: {e}; "
                       f"falling back to thermo.search")

    try:
        target_set = set(elems)
        docs = mpr.materials.thermo.search(
            chemsys=cs_str,
            fields=["material_id", "composition", "energy_per_atom"],
        )
        out = []
        for d in docs:
            comp = Composition(dict(d.composition))
            if {el.symbol for el in comp.elements}.issubset(target_set):
                out.append(PDEntry(comp, d.energy_per_atom * comp.num_atoms,
                                   name=str(d.material_id)))
        return out
    except Exception as e:
        logger.error(f"    Both fetches failed for {cs_str}: {e}")
        return []


def fetch_elemental_endpoint(mpr: MPRester, element: str) -> Optional[PDEntry]:
    """Lowest-energy elemental phase for `element` (used to patch missing endpoints)."""
    try:
        docs = mpr.get_entries_in_chemsys([element])
        if not docs:
            docs = mpr.materials.thermo.search(
                chemsys=element,
                fields=["material_id", "composition", "energy_per_atom"],
            )
    except Exception as e:
        logger.warning(f"  Elemental fetch for {element} failed: {e}")
        return None

    pure = []
    for d in docs:
        comp = Composition(dict(d.composition))
        if {el.symbol for el in comp.elements} == {element}:
            pure.append((d, comp))
    if not pure:
        return None

    best_d, best_comp = min(pure, key=lambda x: x[0].energy_per_atom)
    name = getattr(best_d, "entry_id", None) or getattr(best_d, "material_id", element)
    return PDEntry(best_comp, best_d.energy_per_atom * best_comp.num_atoms,
                   name=f"{element}_endpoint")


def ensure_endpoints(mpr, chemsys, mp_entries):
    """Patch missing elemental endpoints so PhaseDiagram can build."""
    have_pure = set()
    for e in mp_entries:
        e_set = {el.symbol for el in e.composition.elements}
        if len(e_set) == 1:
            have_pure.add(next(iter(e_set)))

    extras = []
    for el in chemsys:
        if el not in have_pure:
            ep = fetch_elemental_endpoint(mpr, el)
            if ep is not None:
                extras.append(ep)
                logger.info(f"    [endpoint patch] added {el}: "
                            f"E/atom = {ep.energy_per_atom:+.4f}")
            else:
                logger.error(f"    [endpoint patch] FAILED for {el} - hull may not build")
    return mp_entries + extras


def print_competing(chemsys, mp_entries, own_entries, target_entry, target_label, e_above):
    cs_str = "-".join(chemsys)
    logger.info(f"\n  -- Hull for {cs_str}  (evaluating {target_label}) --")
    logger.info(f"    MP entries ({len(mp_entries)})")
    logger.info(f"    Dataset entries ({len(own_entries)}, incl. subsystems)")
    logger.info(f"    -> E_above_hull({target_label}) = {e_above:.4f} eV/atom")


# ---- Run ----
df = pd.read_csv(CSV["mace_clean"])
df[H_DIS] = pd.NA
df[H_CHG] = pd.NA

own_by_cs = collect_dataset_entries(df)
n_total   = sum(len(v) for v in own_by_cs.values())
logger.info(f"Loaded {len(df)} rows, {n_total} valid candidates "
            f"across {len(own_by_cs)} chemical systems")

with MPRester(MP_API_KEY) as mpr:
    mp_cache: Dict[tuple, List[PDEntry]] = {}

    for chemsys, own_entries_exact in own_by_cs.items():
        if chemsys not in mp_cache:
            mp_entries = fetch_mp_entries(mpr, chemsys)
            mp_entries = ensure_endpoints(mpr, chemsys, mp_entries)
            mp_cache[chemsys] = mp_entries
        mp_entries = mp_cache[chemsys]

        own_entries_relevant = gather_subsystem_dataset_entries(chemsys, own_by_cs)
        all_entries = mp_entries + [pe for _, _, pe in own_entries_relevant]

        try:
            pd_obj = PhaseDiagram(all_entries)
        except Exception as e:
            logger.warning(f"  PhaseDiagram failed for {'-'.join(chemsys)}: {e}")
            continue

        for idx, state, entry in own_entries_exact:
            hcol = H_DIS if state == "discharged" else H_CHG
            try:
                e_above = pd_obj.get_e_above_hull(entry, on_error="ignore")
            except ValueError:
                e_above = np.nan
            if e_above is None:
                e_above = 0.0

            df.at[idx, hcol] = float(e_above)

            if PRINT_COMPETING_PHASES:
                label = (f"{df.at[idx, 'material_id']}_{state} "
                         f"({entry.composition.reduced_formula})")
                print_competing(chemsys, mp_entries, own_entries_relevant,
                                entry, label, float(e_above))
            else:
                logger.info(f"[{idx}][{state}] {entry.composition.reduced_formula} "
                            f"-> E_above_hull = {float(e_above):.4f} eV/atom")

df.to_csv(CSV["e_above_hull"], index=False)
n_dis = df[H_DIS].notna().sum()
n_chg = df[H_CHG].notna().sum()
logger.info(f"\nWrote {CSV['e_above_hull'].name}  "
            f"({n_dis} discharged, {n_chg} charged values populated)")


## 5.7 Apply thermodynamic stability and voltage criterion

Compute the average intercalation voltage from the (corrected) discharged and charged
energies and the Ca-metal reference, then keep rows where both
`discharged_E_above_hull <= 0.100 eV/atom`, `charged_E_above_hull <= 0.100 eV/atom`, and
`average_voltage ∈ [2.0, 4.5] V`.


In [ ]:
df = pd.read_csv(CSV["e_above_hull"])


def get_ca_count(formula: str) -> int:
    """Number of Ca atoms in a formula string. Examples:
        'CaMn4O6'    -> 1
        'Ca2VFeO8'   -> 2
        'MgO'        -> 0
    """
    m = re.search(r"Ca(\d*)", formula)
    if not m:
        return 0
    return int(m.group(1)) if m.group(1) else 1


df["Ca_count_formula"] = df["new_formula"].apply(get_ca_count)

# Average intercalation voltage:
#   V = -(E_discharged - (E_charged + n_Ca * E_Ca)) / (z * n_Ca)
# with z = 2 (Ca is 2+).
df["average_voltage"] = (
    -(df["discharged_energy_fu_corrected"]
      - (df["charged_energy_fu_corrected"]
         + df["Ca_count_formula"] * E_CA_METAL_PER_ATOM))
    / (2 * df["Ca_count_formula"])
).round(2)

df["average_voltage"] = df["average_voltage"].map("{:.2f}".format)

df.to_csv(CSV["e_hull_voltage"], index=False)
print(f"Wrote {CSV['e_hull_voltage'].name}  ({len(df)} rows)")


In [ ]:
df_stability_voltage = pd.read_csv(CSV["e_hull_voltage"])

# E_above_hull cut-offs
df_meta_stable = df_stability_voltage[
    (df_stability_voltage["charged_E_above_hull"]    <= E_HULL_THRESHOLD) &
    (df_stability_voltage["discharged_E_above_hull"] <= E_HULL_THRESHOLD)
].copy()
df_meta_stable.to_csv(CSV["meta_stable"], index=False)
print(f"Meta-stable (both phases <= {E_HULL_THRESHOLD} eV/atom): "
      f"{len(df_meta_stable)} -> {CSV['meta_stable'].name}")

# Voltage window
df_meta_stable["average_voltage"] = pd.to_numeric(df_meta_stable["average_voltage"],
                                                  errors="coerce")
df_stability_voltage_pass = df_meta_stable[
    df_meta_stable["average_voltage"].between(*VOLTAGE_WINDOW)
]
df_stability_voltage_pass.to_csv(CSV["stability_voltage"], index=False)
print(f"Voltage window {VOLTAGE_WINDOW} V       : "
      f"{len(df_stability_voltage_pass)} -> {CSV['stability_voltage'].name}")


# 6. Criterion 5 — Curation for Ca-mobility analysis

Three curation steps applied to the 433 survivors of Criterion 4:

1. **6.1** Remove structural duplicates within the dataset (cluster by `StructureMatcher`,
   keep the lowest `discharged_E_above_hull`).
2. **6.2** Drop entries whose structure already exists in the Materials Project.
3. **6.3** Manually re-include MP-matched compounds that have not been studied as Ca-cathodes.


## 6.1 Remove structural duplicates within the dataset

For rows sharing the same `new_formula`, cluster by structural similarity (pymatgen
`StructureMatcher`) and keep the entry with the lowest `discharged_E_above_hull` from each
cluster. Rows whose formula appears only once are kept as-is.


In [ ]:
# --- Config local to this cell ---
DUPRM_SUBDIR     = "discharged"
DUPRM_POSCAR     = "POSCAR"
DUPRM_ENERGY_COL = "discharged_E_above_hull"   # tie-breaker within a cluster


@dataclass
class MaterialStructure:
    """One row + its loaded POSCAR structure."""
    idx:       int
    mp_id:     str
    formula:   str
    structure: Optional[Structure] = field(default=None, repr=False)

    @classmethod
    def from_row(cls, idx: int, row: pd.Series, base_dir: Path):
        mp_id   = str(row["material_id"]).strip()
        formula = str(row["new_formula"]).replace(" ", "")
        path    = base_dir / f"{mp_id}_{formula}" / DUPRM_SUBDIR / DUPRM_POSCAR
        if not path.exists():
            logger.warning(f"  [{mp_id}] missing POSCAR: {path}")
            return None
        try:
            structure = Poscar.from_file(str(path)).structure
        except Exception as e:
            logger.error(f"  [{mp_id}] failed to read {path}: {e}")
            return None
        return cls(idx=idx, mp_id=mp_id, formula=formula, structure=structure)


class DuplicateRemover:
    def __init__(self, df, base_dir, energy_col=DUPRM_ENERGY_COL, matcher=None):
        self.df         = df
        self.base_dir   = Path(base_dir)
        self.energy_col = energy_col
        self.matcher    = matcher or StructureMatcher()

    def _formula_groups(self):
        counts = self.df["new_formula"].value_counts()
        dups   = set(counts[counts > 1].index)
        uniq   = set(counts[counts == 1].index)
        return uniq, dups

    def _cluster(self, mats):
        """Greedy single-link: a structure joins the first cluster where any member matches."""
        clusters = []
        for mat in mats:
            for cluster in clusters:
                if any(self.matcher.fit(other.structure, mat.structure) for other in cluster):
                    cluster.append(mat)
                    break
            else:
                clusters.append([mat])
        return clusters

    def _best_in_cluster(self, cluster):
        """Cluster member with the smallest energy_col; NaN-energies pushed to the end."""
        def key(m):
            v = self.df.at[m.idx, self.energy_col]
            return (pd.isna(v), v)
        return min(cluster, key=key).idx

    def process(self):
        if self.energy_col not in self.df.columns:
            raise KeyError(f"Energy column '{self.energy_col}' not found in dataframe")

        unique_formulas, dup_formulas = self._formula_groups()
        keep = []

        # 1. Unique formulas: kept as-is
        unique_mask = self.df["new_formula"].isin(unique_formulas)
        keep.extend(self.df.index[unique_mask].tolist())
        logger.info(f"{unique_mask.sum()} rows with unique formulas (kept as-is); "
                    f"{len(dup_formulas)} formulas appear in multiple rows")

        # 2. Duplicate-formula groups: cluster, then keep best
        n_dup_groups, n_clusters_total = 0, 0
        for formula in sorted(dup_formulas):
            subset = self.df[self.df["new_formula"] == formula]
            mats   = [m for idx, row in subset.iterrows()
                      if (m := MaterialStructure.from_row(idx, row, self.base_dir)) is not None]
            if not mats:
                logger.warning(f"  {formula}: {len(subset)} rows, none loadable - skipping")
                continue
            clusters = self._cluster(mats)
            n_dup_groups     += 1
            n_clusters_total += len(clusters)
            logger.info(f"  {formula}: {len(subset)} entries "
                        f"({len(mats)} loaded) -> {len(clusters)} cluster(s)")
            for cluster in clusters:
                keep.append(self._best_in_cluster(cluster))

        keep_sorted = sorted(set(keep))
        filtered    = self.df.loc[keep_sorted].reset_index(drop=True)
        logger.info(f"Final: {len(filtered)} rows kept (from {len(self.df)}); "
                    f"{n_clusters_total} clusters across {n_dup_groups} duplicate-formula groups")
        return filtered


df         = pd.read_csv(CSV["stability_voltage"])
remover    = DuplicateRemover(df, STRUCTURES_DIR)
df_dedupe  = remover.process()
df_dedupe.to_csv(CSV["duplicates_removed"], index=False)
print(f"\nWrote {CSV['duplicates_removed'].name}  ({len(df_dedupe)} rows)")


## 6.2 Drop entries already present in the Materials Project

For each chemical formula in the dataset, compare each locally generated discharged POSCAR
against every MP polymorph with the same formula (via `StructureMatcher`). Rows whose local
structure matches an MP entry are flagged as already-known and removed; the remainder is the
novel-candidate set.


In [ ]:
@dataclass
class LocalEntry:
    idx:       int
    mp_id:     str
    formula:   str
    structure: Structure


def load_local(idx: int, mp_id: str, formula: str, base_dir: Path) -> Optional[LocalEntry]:
    """Load the canonical POSCAR for one row, or None if missing/unreadable."""
    path = base_dir / f"{mp_id}_{formula}" / DUPRM_SUBDIR / DUPRM_POSCAR
    if not path.exists():
        logger.warning(f"  [{mp_id}] missing {path.name}: {path}")
        return None
    try:
        struct = Poscar.from_file(str(path)).structure
    except Exception as e:
        logger.error(f"  [{mp_id}] failed to read {path}: {e}")
        return None
    return LocalEntry(idx=idx, mp_id=mp_id, formula=formula, structure=struct)


def fetch_mp_structures(mpr: MPRester, formula: str) -> List[Structure]:
    """Fetch all MP polymorphs for `formula`. Returns [] on any failure."""
    try:
        docs = mpr.materials.summary.search(formula=formula, fields=["structure"])
        return [d.structure for d in docs if d.structure is not None]
    except Exception as e:
        logger.warning(f"  MP fetch failed for {formula}: {e}")
        return []


df = pd.read_csv(CSV["duplicates_removed"])
logger.info(f"Loaded {len(df)} rows from {CSV['duplicates_removed'].name}")

df["material_id"] = df["material_id"].astype(str).str.strip()
df["new_formula"] = df["new_formula"].astype(str).str.replace(" ", "", regex=False)

matcher     = StructureMatcher()
matched_idxs: set = set()

n_formulas = df["new_formula"].nunique()
logger.info(f"Querying MP for {n_formulas} unique formulas...")

with MPRester(MP_API_KEY) as mpr:
    for formula, sub in df.groupby("new_formula"):
        if not formula:
            continue
        locals_ = [
            entry for idx, row in sub.iterrows()
            if (entry := load_local(idx, row["material_id"], formula, STRUCTURES_DIR)) is not None
        ]
        if not locals_:
            continue
        mp_structs = fetch_mp_structures(mpr, formula)
        if not mp_structs:
            logger.info(f"  {formula}: {len(locals_)} local, 0 MP -> all novel")
            continue
        n_hit = 0
        for loc in locals_:
            for mp_struct in mp_structs:
                if matcher.fit(loc.structure, mp_struct):
                    matched_idxs.add(loc.idx)
                    n_hit += 1
                    break
        logger.info(f"  {formula}: {len(locals_)} local vs {len(mp_structs)} MP "
                    f"-> {n_hit} match(es)")

df_known = df.loc[sorted(matched_idxs)].reset_index(drop=True)
df_novel = df.drop(index=matched_idxs).reset_index(drop=True)

df_novel.to_csv(CSV["mp_novel"],   index=False)
df_known.to_csv(CSV["mp_matched"], index=False)
logger.info(f"Wrote {len(df_novel)} novel rows  -> {CSV['mp_novel'].name}")
logger.info(f"Wrote {len(df_known)} matched rows -> {CSV['mp_matched'].name}")


## 6.3 Re-include MP-matched compounds not previously studied as Ca cathodes

Of the 31 compounds removed in step 6.2, we manually reviewed each one to identify which had
not been explored as Ca cathodes. The 12 listed below are added back into the novel set.
Note that the `material_id` listed here corresponds to the original queried compound prior to
Ca substitution — not to the matching MP polymorph itself.


In [ ]:
# Material IDs to rescue from MATCHED back into NOVEL (have not been explored as Ca cathodes)
RETAIN_IDS: List[str] = [
    "mp-752490", "mp-1235847", "mp-1044171", "mp-21665",   "mp-1105628", "mp-1013762",
    "mp-1041036", "mp-1013922", "mp-1099034", "mp-21213",  "mp-19287",   "mp-558048",
]

df_matched = pd.read_csv(CSV["mp_matched"])
df_novel   = pd.read_csv(CSV["mp_novel"])

df_matched["material_id"] = df_matched["material_id"].astype(str).str.strip()
df_novel["material_id"]   = df_novel["material_id"].astype(str).str.strip()
retain = set(RETAIN_IDS)

n_matched_before = len(df_matched)
n_novel_before   = len(df_novel)

# 1. Rows to move
move_mask = df_matched["material_id"].isin(retain)
to_move   = df_matched[move_mask]

found_ids   = set(to_move["material_id"])
missing_ids = retain - found_ids
if missing_ids:
    print(f"[warn] {len(missing_ids)} ID(s) not found in {CSV['mp_matched'].name}:")
    for mid in sorted(missing_ids):
        print(f"        {mid}")

# 2. Drop them from matched
df_matched_new = df_matched[~move_mask].reset_index(drop=True)

# 3. Append to novel (skip those already there)
already_in_novel = to_move["material_id"].isin(df_novel["material_id"])
if already_in_novel.any():
    dupes = to_move.loc[already_in_novel, "material_id"].tolist()
    print(f"[warn] skipping {len(dupes)} ID(s) already present in novel CSV: {dupes}")
    to_move = to_move[~already_in_novel]

df_novel_new = pd.concat([df_novel, to_move], ignore_index=True)

# 4. Save
df_matched_new.to_csv(CSV["mp_matched"], index=False)
df_novel_new.to_csv(CSV["mp_novel"],     index=False)

print(f"\nMoved {len(to_move)} row(s) from matched -> novel")
print(f"  {CSV['mp_matched'].name}: {n_matched_before} -> {len(df_matched_new)}")
print(f"  {CSV['mp_novel'].name}:   {n_novel_before} -> {len(df_novel_new)}")

# `mp_novel` is now the curated set used by Criterion 6. We also export to
# CSV["curated"] for downstream clarity.
df_novel_new.to_csv(CSV["curated"], index=False)
print(f"\nCurated set (Criterion 5 output): {len(df_novel_new)} rows -> {CSV['curated'].name}")


## 6.4 Manual exclusion of non-viable Ca pathways

Each remaining structure was visualised to confirm that connected Ca channels exist with a
Ca–Ca hopping distance < 6 Å. Compounds without viable pathways (or already studied as Ca
cathodes in the literature) were dropped manually. The curated input to Criterion 6 is
written to `CSV["curated_for_neb"]`.


In [ ]:
# If you've prepared a manually-curated CSV for NEB, load it here. Otherwise this falls
# back to the full curated set.
if CSV["curated_for_neb"].exists():
    df_curated_for_neb = pd.read_csv(CSV["curated_for_neb"])
else:
    df_curated_for_neb = pd.read_csv(CSV["curated"])
    df_curated_for_neb.to_csv(CSV["curated_for_neb"], index=False)
print(f"Curated set for NEB: {len(df_curated_for_neb)} rows -> {CSV['curated_for_neb'].name}")


# 7. Criterion 6 — Ca Mobility

Three complementary methods evaluate Ca-ion migration barriers:

1. **MACE-NEB** — nudged-elastic-band using the MACE-MP-0 large model.
2. **ORB-v3 NEB** — NEB using the ORB-v3 conservative force field.
3. **Transfer-Learning (TL) model** — ALIGNN-based regression model fine-tuned on Ca NEB data.

The overall Ca migration barrier threshold for candidacy is **< 1000 meV**.


## 7.1 MACE-NEB calculations

For every material in `STRUCTURES_DIR / {mp-id}_* / discharged / neb*`, relax the initial
and final POSCAR, build a 7-image NEB with IDPP interpolation, and write the per-image
energies + the migration barrier to `mace_neb/energies.csv`.


In [ ]:
# Uncomment MACE/ASE/torch imports in Section 1.1 before running.
# import torch
# from ase.io import read, write
# from ase.optimize import QuasiNewton, BFGS
# from ase.mep.neb import NEB
# from mace.calculators import mace_mp


class PoscarReader:
    """Read a POSCAR via ASE, with a fallback for files containing trailing junk."""

    @staticmethod
    def read(path):
        try:
            return read(str(path))
        except AssertionError:
            lines = Path(path).read_text().splitlines()
            num_line = next(
                i for i, ln in enumerate(lines)
                if re.fullmatch(r"\s*\d+(?:\s+\d+)*\s*", ln)
            )
            natoms = sum(map(int, lines[num_line].split()))
            coord_hdr = next(
                i for i in range(num_line + 1, len(lines))
                if lines[i].strip().lower().startswith(("direct", "cart"))
            )
            last = coord_hdr + 1 + natoms
            return read(io.StringIO("\n".join(lines[:last])), format="vasp")


class MACECalcFactory:
    def __init__(self, model_path, device=None, dtype="float64"):
        self.model_path = str(model_path)
        self.device     = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype      = dtype

    def make(self):
        return mace_mp(model=self.model_path, default_dtype=self.dtype, device=self.device)


class NebWorkflow:
    def __init__(self, base_dir, params, calc_factory):
        self.base         = Path(base_dir)
        self.params       = params
        self.calc_factory = calc_factory

    def run(self):
        for folder in self.base.iterdir():
            if not folder.is_dir():
                continue
            discharged = folder / "discharged"
            if not discharged.is_dir():
                continue
            for neb_dir in discharged.glob("neb*"):
                self._process_neb(folder, neb_dir)
        print("MACE-NEB complete.")

    def _process_neb(self, folder, neb_path):
        save_dir  = neb_path / "mace_neb"
        barrier_f = save_dir / "energies.csv"
        if barrier_f.exists():
            return
        save_dir.mkdir(parents=True, exist_ok=True)
        try:
            initial = PoscarReader.read(neb_path / "POSCAR_ini")
            final   = PoscarReader.read(neb_path / "POSCAR_fin")
            for atoms, lbl in ((initial, "initial"), (final, "final")):
                atoms.calc = self.calc_factory.make()
                QuasiNewton(atoms, trajectory=str(save_dir / f"{lbl}.traj")).run(
                    fmax=self.params["fmax"], steps=self.params["max_steps"]
                )
                write(str(save_dir / f"{lbl}_relaxed.vasp"), atoms, "vasp")
            initial = read(str(save_dir / "initial.traj"))
            final   = read(str(save_dir / "final.traj"))
            images  = [initial]
            for _ in range(self.params["images_between"]):
                img = initial.copy(); img.calc = self.calc_factory.make()
                images.append(img)
            final.calc = self.calc_factory.make(); images.append(final)
            neb = NEB(images, parallel=True, k=2.5, method="eb")
            neb.interpolate("idpp")
            BFGS(neb, trajectory=str(save_dir / "neb.traj")).run(
                fmax=self.params["fmax"], steps=self.params["max_steps"]
            )
            energies = []
            for img in images:
                try:
                    energies.append(img.get_potential_energy())
                except Exception:
                    energies.append(None)
            with open(barrier_f, "w", newline="") as fh:
                wr = csv.writer(fh)
                wr.writerow(["Image", "Energy (eV)"])
                for i, e in enumerate(energies):
                    wr.writerow([i, e])
                if None not in energies:
                    wr.writerow([])
                    wr.writerow(["Migration Barrier", max(energies) - min(energies)])
        except Exception as e:
            logging.error(f"{neb_path}: {e}")
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache(); gc.collect()


mace_params = {"id_cutoff": 1_000_000, "images_between": 7, "fmax": 0.05, "max_steps": 1000}
mace_calc   = MACECalcFactory(MACE_MODEL_PATH)
NebWorkflow(NEB_BASE_DIR, mace_params, mace_calc).run()


In [ ]:
# --- Collect MACE-NEB barriers into a single CSV ---
class NebProcessor:
    HEADER = ["mace-neb1 (meV)", "mace-neb2", "mace-neb3", "mace-neb4", "Remarks"]

    def __init__(self, base_path, input_csv, output_csv):
        self.base       = Path(base_path)
        self.input_csv  = Path(input_csv)
        self.output_csv = Path(output_csv)

    def _barriers(self, material_id):
        barriers = ["NA"] * 4
        others   = []
        for sub in self.base.iterdir():
            if not sub.is_dir() or not sub.name.startswith(material_id):
                continue
            disc = sub / "discharged"
            if not disc.is_dir():
                continue
            for neb in disc.iterdir():
                if neb.is_dir() and neb.name.startswith("neb"):
                    try:
                        idx = int(neb.name[3]) - 1
                    except Exception:
                        continue
                    if not 0 <= idx < 4:
                        continue
                    for fname in ("energies.csv", "energy.csv"):
                        f = neb / "mace_neb" / fname
                        if f.exists():
                            try:
                                last = f.read_text().splitlines()[-1]
                                barriers[idx] = str(int(round(float(last.split(",")[-1]) * 1000)))
                            except Exception:
                                pass
                            break
                else:
                    others.append(neb.name)
        return barriers, (";".join(others) or "one")

    def run(self):
        df = pd.read_csv(self.input_csv)
        for col in self.HEADER[:-1]:
            df[col] = None
        df["Remarks"] = None
        for i, row in df.iterrows():
            barriers, remarks = self._barriers(str(row["material_id"]))
            for j, col in enumerate(self.HEADER[:-1]):
                df.loc[i, col] = barriers[j]
            df.loc[i, "Remarks"] = remarks
        df.to_csv(self.output_csv, index=False)
        print(f"Saved MACE-NEB results to {self.output_csv}")


NebProcessor(NEB_BASE_DIR, CSV["curated_for_neb"], CSV["mace_neb"]).run()


In [ ]:
# Copy MACE-NEB results to the labelled CSV used by downstream steps
df_mace_neb = pd.read_csv(CSV["mace_neb"])
df_mace_neb.to_csv(CSV["mace_neb_labels"], index=False)
print(f"MACE-NEB results: {len(df_mace_neb)} compounds -> {CSV['mace_neb_labels'].name}")


## 7.2 ORB-v3 NEB calculations

Repeat the NEB workflow with the ORB-v3 conservative model. ORB-v3 enforces fixed bottom-layer
atoms via `FixAtoms`, the only mechanical difference from the MACE workflow.


In [ ]:
# Uncomment ORB/ASE/torch imports in Section 1.1 before running.
# from orb_models.forcefield import pretrained
# from orb_models.forcefield.calculator import ORBCalculator
# from ase.constraints import FixAtoms


class ORBCalcFactory:
    def __init__(self, precision="float32-high"):
        self.device    = "cuda" if torch.cuda.is_available() else "cpu"
        self.precision = precision

    def make(self):
        return ORBCalculator(pretrained.orb_v3_conservative_inf_omat(
            device=self.device, precision=self.precision
        ))


class ORBNebWorkflow:
    def __init__(self, base_dir, params):
        self.base         = Path(base_dir)
        self.params       = params
        self.calc_factory = ORBCalcFactory()

    def run(self):
        for folder in self.base.glob("mp-*"):
            discharged = folder / "discharged"
            if not discharged.is_dir():
                continue
            for neb_dir in discharged.glob("neb*"):
                self._process(neb_dir)
        print("ORB-v3 NEB complete.")

    def _process(self, neb_path):
        save_dir = neb_path / "ORBV3-neb"
        if (save_dir / "energies.csv").exists():
            return
        if not (neb_path / "POSCAR_ini").exists():
            return
        save_dir.mkdir(parents=True, exist_ok=True)
        try:
            initial = PoscarReader.read(neb_path / "POSCAR_ini")
            final   = PoscarReader.read(neb_path / "POSCAR_fin")
            fixed   = FixAtoms([a.tag > 1 for a in final])
            for atoms, lbl in ((initial, "initial"), (final, "final")):
                atoms.set_constraint(fixed); atoms.calc = self.calc_factory.make()
                BFGS(atoms, trajectory=str(save_dir / f"{lbl}.traj")).run(
                    fmax=self.params["fmax"], steps=self.params["max_steps"]
                )
                write(str(save_dir / f"{lbl}_relaxed.vasp"), atoms, "vasp")
            initial = read(str(save_dir / "initial.traj"))
            final   = read(str(save_dir / "final.traj"))
            images  = [initial]
            for _ in range(self.params["images_between"]):
                img = initial.copy(); img.set_constraint(fixed)
                img.calc = self.calc_factory.make(); images.append(img)
            final.calc = self.calc_factory.make(); images.append(final)
            neb = NEB(images, parallel=True, k=5.0, method="eb")
            neb.interpolate("idpp")
            BFGS(neb, trajectory=str(save_dir / "neb.traj")).run(
                fmax=self.params["fmax"], steps=self.params["max_steps"]
            )
            energies = []
            for i, img in enumerate(images):
                try:
                    energies.append(img.get_potential_energy())
                except Exception:
                    energies.append(None)
                write(str(save_dir / f"neb_opt_{i}.vasp"), img, "vasp")
            with open(save_dir / "energies.csv", "w", newline="") as fh:
                wr = csv.writer(fh)
                wr.writerow(["Image", "Energy (eV)"])
                for i, e in enumerate(energies):
                    wr.writerow([i, e])
                if None not in energies:
                    wr.writerow([])
                    wr.writerow(["Migration Barrier", max(energies) - min(energies)])
        except Exception as e:
            logging.error(f"{neb_path}: {e}")
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache(); gc.collect()


orb_params = {"images_between": 7, "fmax": 0.05, "max_steps": 3000}
ORBNebWorkflow(NEB_BASE_DIR, orb_params).run()


In [ ]:
# --- Collect ORB-v3 barriers into the combined CSV ---
df = pd.read_csv(CSV["mace_neb_labels"])
for col in ["orbv3-neb1 (meV)", "orbv3-neb2", "orbv3-neb3", "orbv3-neb4"]:
    df[col] = None

for i, row in df.iterrows():
    mid  = str(row["material_id"])
    dirs = list(NEB_BASE_DIR.glob(f"{mid}_*/"))
    if not dirs:
        continue
    discharged = dirs[0] / "discharged"
    if not discharged.exists():
        continue
    for n in range(1, 5):
        neb_path = discharged / f"neb{n}"
        if not neb_path.is_dir():
            continue
        target = neb_path / "ORBV3-neb" / "energies.csv"
        if not target.exists():
            continue
        try:
            val = float(pd.read_csv(target).iloc[-1, -1]) * 1000
            col = f"orbv3-neb{n}" if n > 1 else "orbv3-neb1 (meV)"
            df.loc[i, col] = round(val, 1)
        except Exception:
            pass

df.to_csv(CSV["mace_orbv3_labels"], index=False)
print(f"Saved MACE + ORB-v3 labels to {CSV['mace_orbv3_labels']}")


## 7.3 Transfer-Learning (TL) Model

The TL model is an ALIGNN-based graph neural network fine-tuned on DFT-calculated migration
barriers. Structures are converted to JARVIS Atoms JSON, fed to the model, and predictions
are inverse-normalised before being merged with the MACE / ORB-v3 results.

**Input structure pool:** 262 structures with valid migration pathways out of 292 candidates
(30 excluded due to Ca–Ca hopping distance > 6 Å, blocked channels, or prior literature
coverage).


### 7.3.1 Convert POSCARs to JARVIS JSON


In [ ]:
# !pip install --quiet jarvis-tools ase  # uncomment if not installed
# from ase.io import read as ase_read
# from jarvis.core.atoms import Atoms as JarvisAtoms


def poscar_to_jarvis(poscar_path, out_json_path):
    """Read a VASP POSCAR via ASE and write a JARVIS Atoms JSON."""
    atoms = ase_read(str(poscar_path), format="vasp")
    j = JarvisAtoms(
        lattice_mat=atoms.get_cell().tolist(),
        coords=atoms.get_scaled_positions().tolist(),
        elements=atoms.get_chemical_symbols(),
        cartesian=False,
    )
    with open(out_json_path, "w") as f:
        json.dump(j.to_dict(), f, indent=2)


for mp_dir in sorted(TL_NEB_DIR.iterdir()):
    if not mp_dir.is_dir():
        continue
    discharged = mp_dir / "discharged"
    if not discharged.is_dir():
        continue
    for neb in sorted(discharged.iterdir()):
        if not neb.name.startswith("neb"):
            continue
        src  = neb / "POSCAR"
        dest = neb / "POSCAR.json"
        if not src.exists():
            print(f"[{mp_dir.name}/{neb.name}] No POSCAR, skipping")
            continue
        try:
            poscar_to_jarvis(src, dest)
            print(f"[{mp_dir.name}/{neb.name}] -> POSCAR.json")
        except Exception as e:
            print(f"ERROR {src}: {e}")
print("POSCAR -> JARVIS JSON conversion complete.")


### 7.3.2 Build structure_list.json for model input


In [ ]:
structure_list = []

for json_path in glob.glob(str(TL_NEB_DIR / "mp-*" / "discharged" / "neb*" / "POSCAR.json")):
    jid = os.path.relpath(json_path, TL_NEB_DIR)
    with open(json_path) as f:
        jarvis_dict = json.load(f)
    structure_list.append({"jid": jid, "target": 1.0, "atoms": jarvis_dict})

with open(TL_STRUCTURE_JSON, "w") as f:
    json.dump(structure_list, f, indent=2)
print(f"Saved {len(structure_list)} entries to {TL_STRUCTURE_JSON}")


### 7.3.3 TL helper functions (inverse normalisation, table reshaping, candidacy flag)

These helpers were not defined inline in the original notebook — they were imported from
a small companion module. They are reproduced here as a single block so this notebook is
self-contained. If you maintain a separate `tl_utils.py`, you can remove this cell and
replace it with `from tl_utils import process_csv, build_material_tables, assign_candidate`.


In [ ]:
def process_csv(csv_path: Path,
                original_mean: float,
                original_std:  float,
                orig_min:      float,
                orig_max:      float) -> pd.DataFrame:
    """
    Inverse-transform TL predictions from the training-time normalised space back to eV.

    The training pipeline applies (a) min-max scaling to [0, 1] and (b) z-score
    standardisation. To recover physical barriers we undo (b) then (a). The transformed
    DataFrame is also written next to the input CSV with the suffix `_undo_norm.csv`.

    Expected input columns: `id, target, prediction` (target/prediction in normalised space).
    Output columns:         everything from the input plus `target_raw, prediction_raw`.
    """
    df = pd.read_csv(csv_path)
    # 1) undo z-score
    df["target_z"]     = df["target"]     * original_std + original_mean
    df["prediction_z"] = df["prediction"] * original_std + original_mean
    # 2) undo min-max
    df["target_raw"]     = df["target_z"]     * (orig_max - orig_min) + orig_min
    df["prediction_raw"] = df["prediction_z"] * (orig_max - orig_min) + orig_min

    out_path = csv_path.with_name(csv_path.stem + "_undo_norm.csv")
    df.to_csv(out_path, index=False)
    return df


def build_material_tables(json_path: Path, csv_path: Path) -> dict:
    """
    Reshape per-NEB-path predictions into a per-material wide table.

    For each material id we keep at most four NEB paths (neb1 .. neb4) and emit one row
    with columns `materials_id, prediction_raw_neb1, prediction_raw_neb2, ...`.

    The mapping from row->material comes from `json_path` (the structure_list.json used
    as model input); rows in `csv_path` are aligned positionally with entries in that
    JSON. A wide CSV is also written next to `csv_path` with the suffix
    `_wide_by_material.csv`.

    Returns: {"wide_df": <DataFrame>}.
    """
    with open(json_path) as f:
        structure_list = json.load(f)
    df = pd.read_csv(csv_path)

    # `jid` is like 'mp-XXXX/discharged/nebN/POSCAR.json'
    jids = [entry["jid"] for entry in structure_list]
    if len(jids) != len(df):
        # Re-align by jid column if present
        if "jid" in df.columns:
            df = df.set_index("jid").reindex(jids).reset_index()
        else:
            raise ValueError(f"row mismatch: {len(df)} predictions vs {len(jids)} structures")
    else:
        df["jid"] = jids

    def parse_jid(jid: str):
        parts = jid.split("/")
        # ['mp-XXXX', 'discharged', 'nebN', 'POSCAR.json']
        return parts[0], parts[2]                                  # (material_id, neb_label)

    df["materials_id"], df["neb_label"] = zip(*df["jid"].map(parse_jid))

    pred_cols = ["prediction_raw"] if "prediction_raw" in df.columns else ["prediction"]
    wide = df.pivot_table(index="materials_id",
                          columns="neb_label",
                          values=pred_cols,
                          aggfunc="first")
    wide.columns = [f"{val}_{neb}" for val, neb in wide.columns]
    wide = wide.reset_index()

    # Ensure neb1..neb4 columns exist
    for n in range(1, 5):
        c = f"prediction_raw_neb{n}"
        if c not in wide.columns:
            wide[c] = pd.NA

    out_path = csv_path.with_name(csv_path.stem + "_wide_by_material.csv")
    wide.to_csv(out_path, index=False)
    return {"wide_df": wide}


def assign_candidate(row, source: str) -> int:
    """
    Return 1 if any migration barrier from `source` is below the threshold, else 0.

    `source` is one of: 'mace', 'orbv3', 'TL' (case-insensitive).
    """
    src = source.lower()
    if   src == "mace":  cols = ["mace-neb1 (meV)",  "mace-neb2",  "mace-neb3",  "mace-neb4"]
    elif src == "orbv3": cols = ["orbv3-neb1 (meV)", "orbv3-neb2", "orbv3-neb3", "orbv3-neb4"]
    elif src == "tl":    cols = ["TL_neb1 (meV)",    "TL_neb2",    "TL_neb3",    "TL_neb4"]
    else:
        raise ValueError(f"unknown source {source!r}")

    for c in cols:
        if c not in row:
            continue
        v = row[c]
        if pd.isna(v):
            continue
        try:
            if float(v) < NEB_BARRIER_THRESHOLD_meV:
                return 1
        except (TypeError, ValueError):
            continue
    return 0


### 7.3.4 Inverse-transform TL predictions (Interpolated model — `best_model.pt`)


In [ ]:
# Process the best interpolated model predictions
df_inter_best = process_csv(
    TL_INTERP_DIR / "best_test.csv",
    original_mean=TL_MEAN, original_std=TL_STD,
    orig_min=TL_MIN,       orig_max=TL_MAX,
)

# Build per-material wide table
res_inter = build_material_tables(
    json_path=TL_STRUCTURE_JSON,
    csv_path =TL_INTERP_DIR / "best_test_undo_norm.csv",
)
df_inter_wide = res_inter["wide_df"]
print(df_inter_wide.shape)


In [ ]:
# Filter to compounds with at least one barrier below the threshold (1 eV)
thr  = NEB_BARRIER_THRESHOLD_meV / 1000.0   # convert to eV (TL preds are in eV)
mask = (df_inter_wide[["prediction_raw_neb1",
                       "prediction_raw_neb2",
                       "prediction_raw_neb3"]] < thr).any(axis=1)
df_inter_best_cand = df_inter_wide[mask]
print(f"TL candidates (interpolated, best_model.pt): {len(df_inter_best_cand)}")


### 7.3.5 Inverse-transform TL predictions (MP-matching compounds)


In [ ]:
df_mp_tl = process_csv(
    TL_MP_MATCH_DIR / "mp_best_test.csv",
    original_mean=TL_MEAN, original_std=TL_STD,
    orig_min=TL_MIN,       orig_max=TL_MAX,
)

res_mp = build_material_tables(
    json_path=TL_MP_MATCH_DIR / "mp_structures2.json",
    csv_path =TL_MP_MATCH_DIR / "mp_best_test_undo_norm.csv",
)
df_mp_wide = res_mp["wide_df"]

# Filter
mask_mp = (df_mp_wide[["prediction_raw_neb1", "prediction_raw_neb2"]] < thr).any(axis=1)
df_mp_best_cand = df_mp_wide[mask_mp]
print(f"TL candidates (MP-matching): {len(df_mp_best_cand)}")


## 7.4 Analysis of all NEB data


In [ ]:
# --- Load combined MACE + ORB-v3 NEB results ---
df_orb_mace = pd.read_csv(CSV["mace_orbv3_labels"])
print(f"Total compounds with NEB results: {len(df_orb_mace)}")


In [ ]:
# --- Assign 0/1 candidacy flags for MACE and ORB-v3 ---
df = pd.read_csv(CSV["mace_orbv3_labels"])
df["mace_candidate"]  = 0
df["orbv3_candidate"] = 0

for idx, row in df.iterrows():
    df.loc[idx, "mace_candidate"]  = assign_candidate(row, "mace")
    df.loc[idx, "orbv3_candidate"] = assign_candidate(row, "orbv3")

df.to_csv(CSV["candidates_flag"], index=False)
print(f"MACE candidates  : {int(df['mace_candidate'].sum())}")
print(f"ORB-v3 candidates: {int(df['orbv3_candidate'].sum())}")


In [ ]:
# Drop rows where ALL NEB values are NaN (no NEB computed)
df_not_nan = df[~df["mace-neb1 (meV)"].isna()].copy()
df_not_nan.to_csv(CSV["not_nan"], index=False)
print(f"Compounds with at least one NEB value: {len(df_not_nan)}")


In [ ]:
# --- Merge TL predictions into the main candidate table ---


def clean_barrier_df(df_tl):
    """Convert eV prediction columns to meV and rename."""
    base  = ["materials_id"]
    avail = base + [c for c in ["prediction_raw_neb1", "prediction_raw_neb2",
                                 "prediction_raw_neb3", "prediction_raw_neb4"]
                    if c in df_tl.columns]
    df_tl = df_tl[avail].copy()
    for c in avail[1:]:
        df_tl[c] = (df_tl[c] * 1000).round(1)
    rename = {"materials_id": "material_id"}
    for i, c in enumerate(avail[1:], 1):
        rename[c] = f"TL_neb{i} (meV)" if i == 1 else f"TL_neb{i}"
    return df_tl.rename(columns=rename)


df_main = pd.read_csv(CSV["not_nan"])
df_tl1  = clean_barrier_df(pd.read_csv(TL_INTERP_DIR   / "best_test_undo_norm_wide_by_material.csv"))
df_tl2  = clean_barrier_df(pd.read_csv(TL_MP_MATCH_DIR / "mp_best_test_undo_norm_wide_by_material.csv"))

df_barrier = pd.concat([df_tl1, df_tl2]).drop_duplicates("material_id", keep="last")
df_merged  = pd.merge(df_main, df_barrier, on="material_id", how="left")
df_merged.to_csv(CSV["with_tl"], index=False)
print(f"Merged TL barriers. Shape: {df_merged.shape}")


In [ ]:
# --- Assign candidacy flags including TL ---
df = pd.read_csv(CSV["with_tl"])
df["mace_candidate"]  = 0
df["orbv3_candidate"] = 0
df["TL_candidate"]    = 0

for idx, row in df.iterrows():
    df.loc[idx, "mace_candidate"]  = assign_candidate(row, "mace")
    df.loc[idx, "orbv3_candidate"] = assign_candidate(row, "orbv3")
    df.loc[idx, "TL_candidate"]    = assign_candidate(row, "TL")

df.to_csv(CSV["final_candidates"], index=False)
print(f"MACE candidates  : {int(df['mace_candidate'].sum())}")
print(f"ORB-v3 candidates: {int(df['orbv3_candidate'].sum())}")
print(f"TL candidates    : {int(df['TL_candidate'].sum())}")


### 7.4.1 Retrieve spacegroups from MP


In [ ]:
df_all        = pd.read_csv(CSV["final_candidates"])
material_ids  = df_all["material_id"].astype(str).dropna().unique().tolist()
space_map: Dict[str, Tuple[Optional[str], Optional[int]]] = {}

with MPRester(MP_API_KEY) as mpr:
    CHUNK = 200
    for i in range(0, len(material_ids), CHUNK):
        chunk   = material_ids[i:i + CHUNK]
        results = mpr.summary.search(material_ids=chunk, fields=["material_id", "symmetry"])
        for r in results:
            sym = getattr(r, "symmetry", None) or {}
            space_map[r.material_id] = (
                getattr(sym, "symbol", None) if hasattr(sym, "symbol") else sym.get("symbol"),
                getattr(sym, "number", None) if hasattr(sym, "number") else sym.get("number"),
            )

mid_idx = df_all.columns.get_loc("material_id")
df_all.insert(mid_idx + 1, "spacegroup_symbol",
              [space_map.get(m, (None, None))[0] for m in df_all["material_id"].astype(str)])
df_all.insert(mid_idx + 2, "spacegroup_number",
              [space_map.get(m, (None, None))[1] for m in df_all["material_id"].astype(str)])
df_all.to_csv(CSV["with_spacegroup"], index=False)
print(f"Saved final table with spacegroups to {CSV['with_spacegroup']}")


### 7.4.2 Candidate set intersections


In [ ]:
pd.set_option("display.max_columns", None)
df_all = pd.read_csv(CSV["with_spacegroup"])

df_cand_mace  = df_all[df_all["mace_candidate"]  == 1]
df_cand_orbv3 = df_all[df_all["orbv3_candidate"] == 1]
df_cand_TL    = df_all[df_all["TL_candidate"]    == 1]

df_mace_TL    = df_all[(df_all["mace_candidate"]  == 1) & (df_all["TL_candidate"]    == 1)]
df_orbv3_TL   = df_all[(df_all["orbv3_candidate"] == 1) & (df_all["TL_candidate"]    == 1)]
df_mace_orbv3 = df_all[(df_all["mace_candidate"]  == 1) & (df_all["orbv3_candidate"] == 1)]
df_all3       = df_all[(df_all["mace_candidate"]  == 1)
                       & (df_all["orbv3_candidate"] == 1)
                       & (df_all["TL_candidate"]    == 1)]

print("Candidate summary:")
print(f"  MACE                : {len(df_cand_mace)}")
print(f"  ORB-v3              : {len(df_cand_orbv3)}")
print(f"  TL                  : {len(df_cand_TL)}")
print(f"  MACE n TL           : {len(df_mace_TL)}")
print(f"  ORB-v3 n TL         : {len(df_orbv3_TL)}")
print(f"  MACE n ORB-v3       : {len(df_mace_orbv3)}")
print(f"  MACE n ORB-v3 n TL  : {len(df_all3)}")


### 7.4.3 Unique formula count per candidate set


In [ ]:
print("Unique formulas:")
print(f"  MACE          : {len(df_cand_mace['new_formula'].unique())}")
print(f"  ORB-v3        : {len(df_cand_orbv3['new_formula'].unique())}")
print(f"  MACE n ORB-v3 : {len(df_mace_orbv3['new_formula'].unique())}")


### 7.4.4 DFT validation

The final 37 candidates from the MACE, ORB-v3 and TL intersection are validated with DFT
NEB calculations. 
